# Florence-2: Warm-Start vs Fresh Fine-Tuning — Head-to-Head Comparison

| | Model A (Fresh) | Model B (Warm-Start) |
|---|---|---|
| **Base** | `microsoft/Florence-2-large-ft` | `custom_nssf_model` (your 87-sample model) |
| **Training data** | 8,125 NSSF samples | 8,125 NSSF samples (same split) |
| **Strategy** | LoRA from random init | LoRA continuing from learned weights |
| **GPU** | RTX 4090 — `bf16=True` (Ada Lovelace) | RTX 4090 — `bf16=True` (Ada Lovelace) |
| **Hypothesis** | Strong OCR baseline | Faster convergence, better handwriting |

Both models run **sequentially** inside a single `train_florence_experiment()` function.  
VRAM is fully cleared between runs. Expected runtime: ~3–5 hours per model on a 4090.

**Tracking**: Both runs log to the same WandB project — loss curves overlay on the dashboard.

---
### Why `bf16` instead of `fp16` on an RTX 4090
`fp16` (IEEE half-precision) has a narrow dynamic range and can produce `NaN` loss values  
when gradients spike during handwriting-heavy batches. `bf16` (bfloat16) uses the same  
16-bit width but keeps the full 8-bit exponent of fp32, so it never overflows.  
Ada Lovelace has dedicated bf16 tensor cores — using `fp16` instead would leave them idle.


## 0. Install Dependencies

In [ ]:
# ── Disable hf_transfer fast download (env var set by RunPod but package
# may not be installed yet — causes ValueError before anything runs) ────────
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

import subprocess, sys, os

subprocess.run('pip config set global.root-user-action ignore', shell=True, capture_output=True)

# ── Step 1: Upgrade PyTorch for Blackwell GPU (sm_120 / RTX PRO 4500) ────────
# RTX PRO 4500 (Blackwell, sm_120) requires PyTorch ≥ 2.7.0 built with CUDA 12.8.
# The default RunPod image ships with PyTorch 2.x + CUDA 11.x/12.x which only
# supports up to sm_90 (Hopper). Running on Blackwell with an old wheel gives:
#   "CUDA error: no kernel image is available for execution on the device"
# We uninstall the old torch stack first, then pull the CUDA 12.8 wheels.
print('Upgrading PyTorch for Blackwell sm_120 support…')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    capture_output=True
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--root-user-action=ignore',
     'torch>=2.7.0', 'torchvision', 'torchaudio',
     '--index-url', 'https://download.pytorch.org/whl/cu128',
    ],
    check=True
)
# Quick sanity check
import importlib
torch = importlib.import_module('torch')
print(f'✅ PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}')
cap = torch.cuda.get_device_capability()
print(f'   GPU capability: sm_{cap[0]}{cap[1]}  (need sm_120 for Blackwell)')

# ── Step 2: Fix typing_extensions / pydantic-core conflict ───────────────────
# wandb ≥ 0.18 → pydantic ≥ 2.12 → pydantic-core → needs typing_extensions ≥ 4.12.0
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--root-user-action=ignore',
     'typing_extensions>=4.12.0', 'pydantic>=2.0', 'pydantic-core>=2.0',
    ],
    check=True
)
print('✅ typing_extensions / pydantic upgraded')

# ── Step 3: Install everything else ──────────────────────────────────────────
pkgs = [
    'transformers==4.49.0',
    'peft>=0.10.0',
    'accelerate>=0.27.0',
    'bitsandbytes>=0.43.0',
    'einops', 'timm', 'sentencepiece', 'protobuf',
    'rapidfuzz', 'pillow', 'tqdm',
    'opencv-python-headless',
    'wandb',
    'matplotlib',
    'gdown',
    'py7zr',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--root-user-action=ignore'] + pkgs,
    check=True
)
print('✅ All dependencies installed')
print()
print('⚠  IMPORTANT: Kernel restart required after this cell.')
print('   Kernel → Restart Kernel, then run from cell 2 onwards.')
print('   (The old torch is cached in memory until restart.)')


## 0b. Download & Extract Data from Google Drive

In [ ]:
import os, glob, shutil
import gdown

# ── Google Drive file IDs ──────────────────────────────────────────────────
# These are set from your original notebook — change if you re-upload.
DATASET_FILE_ID = '1WMF_rIQ3nAJfFNqaNNXbDM0GOMLSZhvP'   # nssf_dataset.7z
MODEL_FILE_ID   = '1GvkoqBX9aS3sxPM7nsO5YFVbb9-ML50m'   # custom_nssf_model.7z

# ── Local destination paths ────────────────────────────────────────────────
# Keep /workspace as hard-coded base — this cell runs before BASE_DIR is set
# in the config cell, and /workspace is always correct on RunPod.
_base = '/workspace' if os.path.exists('/workspace') else os.path.expanduser('~')

downloads = {
    DATASET_FILE_ID: os.path.join(_base, 'nssf_dataset.7z'),
    MODEL_FILE_ID:   os.path.join(_base, 'custom_nssf_model.7z'),
}

for fid, dest in downloads.items():
    if not fid:
        print(f'⚠  File ID empty — skipping {os.path.basename(dest)}')
        continue
    if os.path.exists(dest):
        size_gb = os.path.getsize(dest) / 1e9
        print(f'✅ Already downloaded: {os.path.basename(dest)} ({size_gb:.2f} GB) — skipping')
        continue
    print(f'Downloading {os.path.basename(dest)} from Google Drive…')
    gdown.download(f'https://drive.google.com/uc?id={fid}', dest, quiet=False)
    size_gb = os.path.getsize(dest) / 1e9
    print(f'✅ Saved: {dest} ({size_gb:.2f} GB)')

print()
print('Download step complete.')


In [ ]:
import zipfile, tarfile

def smart_extract(archive_path, dest_folder):
    """
    Extracts .7z / .zip / .tar.gz into dest_folder.
    Detects the actual format by magic bytes — never trusts the file extension.
    Always extracts INTO dest_folder (never dumps files into the parent dir).
    """
    if not os.path.exists(archive_path):
        print(f'⚠  Not found: {archive_path}')
        return False

    os.makedirs(dest_folder, exist_ok=True)
    size_mb = os.path.getsize(archive_path) / 1e6

    with open(archive_path, 'rb') as f:
        magic = f.read(8)

    is_7z  = magic[:6] == b'7z\xbc\xaf\x27\x1c'
    is_zip = magic[:2] == b'PK'
    is_tar = magic[:2] == b'\x1f\x8b'

    print(f'Extracting {os.path.basename(archive_path)} ({size_mb:.0f} MB) → {dest_folder}')

    if is_7z:
        print('  Format: 7z')
        try:
            import py7zr
            with py7zr.SevenZipFile(archive_path, mode='r') as z:
                z.extractall(path=dest_folder)
            return True
        except Exception as e:
            print(f'  py7zr failed: {e}')
            return False

    elif is_zip:
        print('  Format: zip')
        with zipfile.ZipFile(archive_path, 'r') as z:
            members = z.namelist()
            top_levels = set(m.split('/')[0] for m in members if m.strip('/'))
            has_wrapper = (len(top_levels) == 1 and
                           any(m.endswith('/') for m in members))
            if has_wrapper:
                parent = os.path.dirname(dest_folder)
                z.extractall(parent)
                inner = os.path.join(parent, list(top_levels)[0])
                if inner != dest_folder and os.path.exists(inner):
                    os.rename(inner, dest_folder)
            else:
                z.extractall(dest_folder)
        return True

    elif is_tar:
        print('  Format: tar.gz')
        with tarfile.open(archive_path, 'r:gz') as t:
            t.extractall(dest_folder)
        return True

    else:
        print(f'  Unknown format (magic: {magic[:6].hex()}) — trying zip…')
        try:
            with zipfile.ZipFile(archive_path, 'r') as z:
                z.extractall(dest_folder)
            return True
        except Exception as e:
            print(f'  Failed: {e}')
            return False


# ── Extract dataset ────────────────────────────────────────────────────────
_base          = '/workspace' if os.path.exists('/workspace') else os.path.expanduser('~')
DATASET_DIR    = os.path.join(_base, 'nssf_dataset')
DATASET_ARCHIVE = os.path.join(_base, 'nssf_dataset.7z')

img_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
    imgs = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(img_exts)]
    # Dataset has a double-nested subdirectory — check one level deeper too
    if not imgs:
        inner = os.path.join(DATASET_DIR, 'nssf_dataset')
        if os.path.exists(inner):
            imgs = [f for f in os.listdir(inner) if f.lower().endswith(img_exts)]
    print(f'✅ Dataset already extracted: {DATASET_DIR}')
    print(f'   {len(imgs)} images found')
else:
    ok = smart_extract(DATASET_ARCHIVE, DATASET_DIR)
    if ok:
        imgs = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(img_exts)]
        if not imgs:
            inner = os.path.join(DATASET_DIR, 'nssf_dataset')
            if os.path.exists(inner):
                imgs = [f for f in os.listdir(inner) if f.lower().endswith(img_exts)]
        print(f'✅ Extracted — {len(imgs)} images in {DATASET_DIR}')
    else:
        print('❌ Dataset extraction failed — check the archive')

# ── Extract custom Florence model ─────────────────────────────────────────
MODEL_DIR     = os.path.join(_base, 'custom_nssf_model')
MODEL_ARCHIVE = os.path.join(_base, 'custom_nssf_model.7z')

if os.path.exists(MODEL_DIR) and len(os.listdir(MODEL_DIR)) > 0:
    model_files = os.listdir(MODEL_DIR)
    print(f'\n✅ Custom model already extracted: {MODEL_DIR}')
    print(f'   {len(model_files)} files: {model_files[:6]}')
else:
    ok = smart_extract(MODEL_ARCHIVE, MODEL_DIR)
    if ok:
        model_files = os.listdir(MODEL_DIR)
        print(f'\n✅ Model extracted — {len(model_files)} files in {MODEL_DIR}')
        print(f'   Contents: {model_files[:8]}')
    else:
        print('\n⚠  Model extraction failed — training will load from HuggingFace instead')

print('\n✅ Data setup complete')


## 1. Imports & Device

In [ ]:
import os, gc, json, time, math, random, re, shutil, copy
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'  # prevent hf_transfer crash if package not yet installed

import numpy as np
import torch
import wandb
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor, AutoModelForCausalLM,
    TrainingArguments, Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability()
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {vram:.1f} GB')
    print(f'Compute: sm_{cap[0]}{cap[1]}')
    # Recommended batch size for Florence-2-large
    if vram < 20:
        BS_REC = 1
    elif vram < 30:
        BS_REC = 2
    else:
        BS_REC = 4
    print(f'Recommended BATCH_SIZE (large model): {BS_REC}')


## 2. Configuration

In [ ]:
IS_RUNPOD      = os.path.exists('/workspace')
BASE_DIR       = '/workspace' if IS_RUNPOD else os.path.expanduser('~')

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_JSON         = os.path.join(BASE_DIR, 'ultimate_training_dataset.json')
IMAGE_BASE_DIR    = BASE_DIR
CUSTOM_MODEL_DIR  = os.path.join(BASE_DIR, 'custom_nssf_model')
FLORENCE_BASE     = 'microsoft/Florence-2-large-ft'

OUT_FRESH         = os.path.join(BASE_DIR, 'florence2_fresh_8k')
OUT_WARMSTART     = os.path.join(BASE_DIR, 'florence2_warmstart_8k')
FINAL_FRESH       = os.path.join(BASE_DIR, 'florence2_fresh_8k_final')
FINAL_WARMSTART   = os.path.join(BASE_DIR, 'florence2_warmstart_8k_final')

for d in [OUT_FRESH, OUT_WARMSTART, FINAL_FRESH, FINAL_WARMSTART]:
    os.makedirs(d, exist_ok=True)

TASK_TOKEN = '<Extract_NSSF_Data>'

# ── Training hyper-parameters ─────────────────────────────────────────────
# RTX PRO 4500: 33.7 GB VRAM. Florence-2-large weights ~6 GB in fp32.
# With LoRA + grad-checkpointing, peak usage sits ~10–14 GB → headroom to spare.
BATCH_SIZE    = 4        # RTX PRO 4500 (34 GB): 4 is safe for large model
GRAD_ACCUM    = 2        # effective batch = 4 × 2 = 8 (unchanged)
NUM_EPOCHS    = 5
LR            = 1e-4
WARMUP_STEPS  = 100
SAVE_STEPS    = 500
LOG_STEPS     = 10
MAX_INPUT_LEN = 512
MAX_LABEL_LEN = 640  # covers 100% of dataset (max ground truth = 527 tokens)
VAL_SPLIT     = 0.05

# ── Precision ─────────────────────────────────────────────────────────────
# RTX PRO 4500 (Blackwell) has dedicated bf16 tensor cores — same as Ada Lovelace.
# bf16 = same speed as fp16 but with fp32's dynamic range → no NaN gradients.
# Blackwell natively supports bf16; keep USE_BF16=True.
# Change to False if running on an older GPU (Turing/Volta) that lacks bf16.
USE_BF16      = True

# ── LoRA ──────────────────────────────────────────────────────────────────
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ['q_proj','k_proj','v_proj','out_proj','fc1','fc2','linear','Conv2d']

# ── Weights & Biases ──────────────────────────────────────────────────────
WANDB_API_KEY = ''
WANDB_PROJECT = 'NSSF-Florence2-OCR'
WANDB_ENTITY  = ''
USE_WANDB     = True

if USE_WANDB:
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY, relogin=True)
    else:
        wandb.login()
    print(f'✅ WandB logged in  →  https://wandb.ai/{WANDB_ENTITY or "me"}/{WANDB_PROJECT}')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('ℹ  WandB disabled')

steps_ep = math.ceil(8125 * (1 - VAL_SPLIT) / (BATCH_SIZE * GRAD_ACCUM))
print(f'\nGPU config   : {"bf16" if USE_BF16 else "fp16"}')
print(f'Batch (eff.) : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'Steps/epoch  : {steps_ep}  |  Total steps: {steps_ep*NUM_EPOCHS}')


## 3. Inspect the Custom 87-Sample Model

Before training, we need to know:
1. Is it a **merged** model (safe to add LoRA directly) or a **PEFT adapter** (need to merge first)?
2. Does it already have `<Extract_NSSF_Data>` in its tokenizer?
3. How many token embeddings does it have vs what the new processor expects?

This cell answers all three questions automatically.


In [ ]:
def inspect_model_dir(path):
    """Diagnose a saved model directory before loading."""
    if not os.path.exists(path):
        print(f'❌ Directory not found: {path}')
        return None

    files = os.listdir(path)
    print(f'Files in {os.path.basename(path)}/:')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(path, f))
        print(f'  {f:<40} {size/1e6:>8.2f} MB')

    # Detect PEFT adapter vs merged model
    is_peft_adapter = 'adapter_config.json' in files
    is_merged       = any(f.endswith('.safetensors') or f == 'pytorch_model.bin'
                          for f in files)

    print()
    if is_peft_adapter:
        print('🔍 Type : PEFT adapter (has adapter_config.json)')
        print('   → Will merge weights before applying new LoRA')
    elif is_merged:
        print('🔍 Type : Merged / full model weights')
        print('   → Can apply new LoRA directly')
    else:
        print('⚠  Type : Unknown — may need manual inspection')

    # Check tokenizer for task token
    tok_path = os.path.join(path, 'tokenizer.json')
    if os.path.exists(tok_path):
        tok_data = json.load(open(tok_path))
        vocab    = tok_data.get('model', {}).get('vocab', {})
        added    = [t['content'] for t in tok_data.get('added_tokens', [])]
        has_task_token = TASK_TOKEN in added or TASK_TOKEN in vocab
        print(f'   Task token {TASK_TOKEN}: {"✅ found" if has_task_token else "❌ NOT found (will be added)"}')
        print(f'   Vocab size: {len(vocab) + len(added)}')
    else:
        print('   ⚠  tokenizer.json not found')
        has_task_token = False

    # Config
    cfg_path = os.path.join(path, 'config.json')
    if os.path.exists(cfg_path):
        cfg = json.load(open(cfg_path))
        print(f'   Model type    : {cfg.get("model_type", "unknown")}')
        print(f'   Vocab size    : {cfg.get("text_config", {}).get("vocab_size", cfg.get("vocab_size", "?"))}')

    return {'is_peft_adapter': is_peft_adapter, 'has_task_token': has_task_token}

print('=== Inspecting custom_nssf_model ===')
custom_info = inspect_model_dir(CUSTOM_MODEL_DIR)
print()
print('=== Inspecting Florence-2-large-ft (remote config) ===')
print('  Will be downloaded from HuggingFace Hub on first use')


## 4. Shared Dataset Class & Collator

In [ ]:
class NSSFDocumentDataset(Dataset):
    """
    Reads ultimate_training_dataset.json (Qwen/LLaVA messages format) and
    converts each entry to a Florence-2 (prompt, answer, PIL_image) tuple.

    Automatically strips the 365-token Qwen instruction from every user
    message and replaces it with the compact task token <Extract_NSSF_Data>.
    Your JSON file is never modified.
    """
    LONG_PROMPT_SIGNALS = (
        'You are an expert data extraction assistant',
        'You are an expert document OCR assistant',
    )

    def __init__(self, records, image_base_dir,
                 task_token=TASK_TOKEN, augment=False):
        self.records        = records
        self.image_base_dir = image_base_dir
        self.task_token     = task_token
        self.aug            = None

        if augment:
            try:
                from torchvision import transforms
                self.aug = transforms.Compose([
                    transforms.RandomApply([
                        transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                               saturation=0.2, hue=0.05)
                    ], p=0.5),
                    transforms.RandomApply([
                        transforms.RandomRotation(
                            degrees=3,
                            interpolation=transforms.InterpolationMode.BILINEAR)
                    ], p=0.3),
                    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
                ])
            except ImportError:
                print('torchvision not available — augmentation disabled')

    def _is_long_prompt(self, t):
        return any(s in t for s in self.LONG_PROMPT_SIGNALS)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        e      = self.records[idx]
        prompt = (self.task_token if self._is_long_prompt(e['messages'][0]['content'])
                  else e['messages'][0]['content'].replace('<image>','').strip()
                       or self.task_token)
        answer = e['messages'][1]['content'].strip()
        image  = Image.open(
            os.path.join(self.image_base_dir, e['images'][0])
        ).convert('RGB')
        if self.aug:
            image = self.aug(image)
        return prompt, answer, image


print('NSSFDocumentDataset defined ✅')


def make_collate_fn(processor):
    """Returns a collate_fn bound to a specific processor instance.

    Why max_length / truncation are NOT passed to processor():
    ──────────────────────────────────────────────────────────
    Florence-2's processor subtracts image_seq_length (577 tokens) from
    max_length before tokenising the text:
        max_length -= self.image_seq_length   # processing_florence2.py:264
    Passing max_length=512 → 512-577 = -65 → OverflowError in the Rust
    tokenizer ('can't convert negative int to unsigned').

    The text input is always <Extract_NSSF_Data> (1 token) — nothing to
    truncate. max_length/truncation are still applied to labels, which
    never receive image tokens.
    """
    def _collate(batch):
        prompts, answers, images = zip(*batch)
        inputs = processor(
            text=list(prompts),
            images=list(images),
            return_tensors='pt',
            padding=True,
            # NO max_length / truncation — see docstring above
        )
        labels = processor.tokenizer(
            list(answers),
            return_tensors='pt',
            padding=True,
            max_length=MAX_LABEL_LEN,
            truncation=True,
            return_attention_mask=False,
        ).input_ids
        labels[labels == processor.tokenizer.pad_token_id] = -100
        inputs['labels'] = labels
        return inputs
    return _collate


print('make_collate_fn defined ✅')


## 5. Shared Train / Val Split

In [ ]:
print('Loading dataset…')
with open(DATA_JSON) as f:
    raw_data = json.load(f)
print(f'Total entries: {len(raw_data)}')

# IMPORTANT: both models use the IDENTICAL split so comparison is valid
random.seed(42)
shuffled = raw_data.copy()
random.shuffle(shuffled)

n_val         = max(1, int(len(shuffled) * VAL_SPLIT))
val_records   = shuffled[:n_val]
train_records = shuffled[n_val:]
print(f'Train: {len(train_records)}  |  Val: {len(val_records)}')
print('✅ Same split will be used for BOTH models')


## 6. Shared `TrainingArguments` Factory & `train_florence_experiment()`

In [ ]:
def make_lora_config():
    return LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        task_type='CAUSAL_LM', bias='none', inference_mode=False,
    )


def train_florence_experiment(
    base_model_path,
    output_dir,
    checkpoint_dir,
    run_name,
    strategy,           # 'fresh' or 'warmstart'
    train_records,
    val_records,
    resume=False,
):
    """
    Complete training pipeline for one Florence-2 experiment run.

    Handles both fresh and warm-start strategies internally:
      fresh     → load Florence-2-large-ft, add LoRA, train
      warmstart → load custom_nssf_model (auto-detects PEFT adapter vs merged),
                  merge if needed, init task token, add fresh LoRA, train

    VRAM is fully cleared at the end — safe to call twice in sequence.
    """
    print(f'\n{"="*60}')
    print(f'  RUN  : {run_name}')
    print(f'  BASE : {base_model_path}')
    print(f'  OUT  : {output_dir}')
    print(f'  BF16 : {USE_BF16}')
    print(f'{"="*60}\n')

    # ── WandB init ────────────────────────────────────────────────────────
    if USE_WANDB:
        wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY or None,
            name=run_name,
            tags=[strategy, 'florence2-large-ft', '8k-dataset', '4090'],
            config={
                'base_model':    base_model_path,
                'strategy':      strategy,
                'dataset_size':  len(train_records),
                'epochs':        NUM_EPOCHS,
                'batch_size':    BATCH_SIZE * GRAD_ACCUM,
                'lr':            LR,
                'lora_r':        LORA_R,
                'lora_alpha':    LORA_ALPHA,
                'precision':     'bf16' if USE_BF16 else 'fp16',
                'max_input_len': MAX_INPUT_LEN,
                'max_label_len': MAX_LABEL_LEN,
            },
            resume='allow',
        )
        print(f'✅ WandB run started → {wandb.run.url}')

    # ── Load processor ────────────────────────────────────────────────────
    proc = AutoProcessor.from_pretrained(base_model_path, trust_remote_code=True)

    task_token_is_new = TASK_TOKEN not in proc.tokenizer.additional_special_tokens
    if task_token_is_new:
        proc.tokenizer.add_special_tokens({'additional_special_tokens': [TASK_TOKEN]})
        print(f'✅ Registered {TASK_TOKEN} in tokenizer')
    else:
        print(f'✅ {TASK_TOKEN} already in tokenizer')

    # ── Load model ────────────────────────────────────────────────────────
    if strategy == 'warmstart':
        # Detect PEFT adapter vs merged model
        files = os.listdir(base_model_path) if os.path.isdir(base_model_path) else []
        is_peft_adapter = 'adapter_config.json' in files

        if is_peft_adapter:
            print('Detected PEFT adapter — merging into Florence base first…')
            base = AutoModelForCausalLM.from_pretrained(
                FLORENCE_BASE, trust_remote_code=True, torch_dtype=torch.float32,
            )
            base.resize_token_embeddings(len(proc.tokenizer))
            from peft import PeftModel
            model = PeftModel.from_pretrained(base, base_model_path).merge_and_unload()
            print('✅ Adapter merged into base weights')
        else:
            print('Loading warm-start as merged model…')
            model = AutoModelForCausalLM.from_pretrained(
                base_model_path, trust_remote_code=True, torch_dtype=torch.float32,
            )
            print('✅ Warm-start model loaded')
    else:
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path, trust_remote_code=True, torch_dtype=torch.float32,
        )
        print('✅ Fresh base model loaded')

    # ── Resize embeddings & warm-init task token ──────────────────────────
    # FIX: resize_token_embeddings() must be called after any tokenizer changes
    model.resize_token_embeddings(len(proc.tokenizer))

    if task_token_is_new:
        # Mean-of-vocab init: places the new token inside the existing embedding
        # manifold rather than at a random out-of-distribution point
        token_id = proc.tokenizer.convert_tokens_to_ids(TASK_TOKEN)
        with torch.no_grad():
            mean_emb = model.get_input_embeddings().weight[:-1].mean(dim=0)
            model.get_input_embeddings().weight[token_id] = mean_emb
        print(f'✅ Task token embedding warm-initialised from vocab mean')

    # FIX: use_cache=False is required when gradient_checkpointing=True
    model.config.use_cache = False

    # ── Apply LoRA ────────────────────────────────────────────────────────
    model = get_peft_model(model, make_lora_config())
    # FIX: enable_input_require_grads() required for PEFT + grad-checkpointing
    model.enable_input_require_grads()
    model.print_trainable_parameters()

    # ── Build datasets & collator ─────────────────────────────────────────
    train_ds = NSSFDocumentDataset(train_records, IMAGE_BASE_DIR, augment=True)
    val_ds   = NSSFDocumentDataset(val_records,   IMAGE_BASE_DIR, augment=False)
    collate  = make_collate_fn(proc)

    # ── TrainingArguments ─────────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=checkpoint_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=WARMUP_STEPS,
        weight_decay=0.01,
        # ── RTX 4090: bf16 uses Ada Lovelace tensor cores ─────────────────
        # fp16 leaves them idle; bf16 is the same speed with no NaN risk
        bf16=USE_BF16,
        fp16=not USE_BF16,   # falls back to fp16 on non-Ampere GPUs
        gradient_checkpointing=True,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        evaluation_strategy='steps',
        eval_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=LOG_STEPS,
        remove_unused_columns=False,
        optim='adamw_torch',
        seed=42,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to='wandb' if USE_WANDB else 'none',
        run_name=run_name,
    )

    # ── Train ─────────────────────────────────────────────────────────────
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate,
    )

    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    elapsed = time.time() - t0
    print(f'\n✅ Training complete in {elapsed/3600:.2f} hrs ({elapsed/60:.1f} min)')

    # ── Save ──────────────────────────────────────────────────────────────
    # Merge LoRA delta matrices into base weights before saving.
    # trainer.save_model() alone saves only the adapter; reloading it
    # causes a vocab size mismatch (51290 vs 51289 embeddings).
    print('Merging LoRA weights into base model…')
    merged = trainer.model.merge_and_unload()
    # Fix vision_config.model_type — PEFT merge can corrupt this field,
    # causing "AssertionError: only DaViT is supported" on next load.
    merged.config.vision_config.model_type = 'davit'
    merged.save_pretrained(output_dir)
    proc.save_pretrained(output_dir)
    print(f'✅ Merged model saved → {output_dir}')

    # ── Log summary to WandB ──────────────────────────────────────────────
    if USE_WANDB:
        wandb.summary['train_time_hrs']  = round(elapsed / 3600, 2)
        wandb.summary['final_eval_loss'] = trainer.state.best_metric
        wandb.finish()
        print('✅ WandB run closed')

    # ── VRAM cleanup ─────────────────────────────────────────────────────
    # Delete every reference before the next run loads — otherwise the 4090
    # holds both model graphs in VRAM simultaneously during the handoff.
    best_metric = trainer.state.best_metric
    del model, trainer, proc, train_ds, val_ds, collate
    gc.collect()
    torch.cuda.empty_cache()
    mem_free = torch.cuda.mem_get_info()[0] / 1e9
    print(f'🧹 VRAM cleared — {mem_free:.1f} GB free')
    print()
    return {'run_name': run_name, 'elapsed_hrs': elapsed/3600,
            'best_eval_loss': best_metric, 'output_dir': output_dir}


print('train_florence_experiment() defined ✅')
print(f'  Precision: {"bf16" if USE_BF16 else "fp16"}')
print(f'  Batch    : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} (eff.)')


---
## 7. Run Both Experiments Sequentially

Single cell to kick off overnight training.  
Run it, close the laptop, check WandB in the morning.

**Expected timeline on an RTX 4090:**
- Model A (Fresh)      — ~3–4 hours
- VRAM clear + reload  — ~2 minutes
- Model B (Warm-Start) — ~3–4 hours


In [ ]:
# ── Safety: close any WandB run left open from a previous crashed attempt ─────
# If the last run errored after wandb.init() but before wandb.finish(),
# the run stays "open". Calling wandb.init() again without finishing first
# creates a duplicate run. This guard closes it cleanly.
try:
    if wandb.run is not None:
        print(f'⚠  Closing orphaned WandB run: {wandb.run.name}')
        wandb.finish(exit_code=1)   # exit_code=1 marks it as failed/interrupted
        print('✅ Orphaned run closed')
except Exception:
    pass  # wandb not initialised yet — no-op

# ══════════════════════════════════════════════════════════════════════════
# OVERNIGHT TRAINING — runs both experiments back to back
# Set RESUME_A / RESUME_B = True if a run was interrupted mid-way
# ══════════════════════════════════════════════════════════════════════════

RESUME_A = False
RESUME_B = False

results = []

# ── Model A: Fresh ────────────────────────────────────────────────────────
result_A = train_florence_experiment(
    base_model_path = FLORENCE_BASE,
    output_dir      = FINAL_FRESH,
    checkpoint_dir  = OUT_FRESH,
    run_name        = 'A-fresh-florence2-large-8k',
    strategy        = 'fresh',
    train_records   = train_records,
    val_records     = val_records,
    resume          = RESUME_A,
)
results.append(result_A)

# ── Model B: Warm-Start ───────────────────────────────────────────────────
result_B = train_florence_experiment(
    base_model_path = CUSTOM_MODEL_DIR,
    output_dir      = FINAL_WARMSTART,
    checkpoint_dir  = OUT_WARMSTART,
    run_name        = 'B-warmstart-florence2-large-8k',
    strategy        = 'warmstart',
    train_records   = train_records,
    val_records     = val_records,
    resume          = RESUME_B,
)
results.append(result_B)

# ── Summary ───────────────────────────────────────────────────────────────
print('🎉 BOTH RUNS COMPLETE')
print()
print(f'{"Run":<30} {"Hours":>7} {"Best Val Loss":>14}')
print('-' * 55)
for r in results:
    print(f'{r["run_name"]:<30} {r["elapsed_hrs"]:>7.2f} {str(r["best_eval_loss"]):>14}')
print()
print('Next step: run the Evaluation cells below (Section 9 onwards)')
if USE_WANDB:
    print(f'WandB dashboard: https://wandb.ai/{WANDB_ENTITY or "me"}/{WANDB_PROJECT}')


---
## 8. (Optional) Resume a Single Run

If only one run was interrupted, call `train_florence_experiment()` directly  
with `resume=True` instead of re-running the full sequential cell above.

```python
# Example: resume only Model B
result_B = train_florence_experiment(
    base_model_path = CUSTOM_MODEL_DIR,
    output_dir      = FINAL_WARMSTART,
    checkpoint_dir  = OUT_WARMSTART,
    run_name        = 'B-warmstart-florence2-large-8k',
    strategy        = 'warmstart',
    train_records   = train_records,
    val_records     = val_records,
    resume          = True,   # ← picks up from latest checkpoint in OUT_WARMSTART
)
```


In [ ]:
# This cell intentionally left empty.
# Model B now runs inside train_florence_experiment() called from Cell 16.
# See Section 8 above for how to resume a single interrupted run.


---
## 9. Evaluation Engine
> **Run all cells below after both models finish training.**


In [ ]:
from peft import PeftConfig, PeftModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import fuzz
from collections import Counter

def load_eval_model(model_dir, device):
    """
    Load a fine-tuned Florence-2 model for evaluation.

    dtype fix: merge_and_unload() can leave mixed dtypes when the adapter
    has fp16 Conv2d weights (vision encoder LoRA). .half() after merge
    forces everything to fp16 consistently.
    Inference functions cast pixel_values to the model dtype before
    calling generate() — the processor always returns fp32 pixel_values.
    """
    proc  = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    files = os.listdir(model_dir)

    if 'adapter_config.json' in files:
        print(f'  Detected PEFT adapter — loading base + merging…')
        peft_cfg  = PeftConfig.from_pretrained(model_dir)
        base_name = peft_cfg.base_model_name_or_path
        base = AutoModelForCausalLM.from_pretrained(
            base_name, trust_remote_code=True, torch_dtype=torch.float16,
        )
        base.resize_token_embeddings(len(proc.tokenizer))
        model = PeftModel.from_pretrained(base, model_dir).merge_and_unload()
        # Force uniform fp16 — adapter Conv2d weights may survive merge in a
        # different dtype causing "Input type (float) and bias (c10::Half)"
        model = model.half()
    else:
        print(f'  Loading merged model…')
        # Load config first and enforce model_type = 'davit' before the model
        # is constructed — the assertion fires inside __init__ before weights
        # are even loaded, so patching config.json on disk beforehand is the
        # only reliable fix.
        from transformers import AutoConfig
        import json as _json, os as _os
        cfg_path = _os.path.join(model_dir, 'config.json')
        if _os.path.exists(cfg_path):
            cfg_data = _json.load(open(cfg_path))
            if cfg_data.get('vision_config', {}).get('model_type') != 'davit':
                cfg_data['vision_config']['model_type'] = 'davit'
                _json.dump(cfg_data, open(cfg_path, 'w'), indent=2)
                print(f'  ⚠ Fixed vision_config.model_type → davit in config.json')
        model = AutoModelForCausalLM.from_pretrained(
            model_dir, trust_remote_code=True, torch_dtype=torch.float16,
        )


    model = model.to(device).eval()
    print(f'  Model dtype : {next(model.parameters()).dtype}')
    return proc, model


def normalize(v):
    return ' '.join(str(v).lower().split()) if v is not None else ''

def cer(ref, hyp):
    r, h = normalize(ref), normalize(hyp)
    if not r: return 0.0 if not h else 1.0
    return min(Levenshtein.distance(r, h) / len(r), 1.0)

def robust_parse(raw):
    """
    4-stage JSON recovery for Florence-2 OCR output.

    Stage 1 — standard json.loads() + recursive key whitespace cleanup
    Stage 2 — fix trailing commas, bare string entries (keys without
               values like "Postal_Address_Box 35-50315"), control chars
    Stage 3 — truncation recovery: cut at last complete key-value pair
               and close the object
    Stage 4 — regex salvage of all flat "Key": "Value" string pairs
               even when Table arrays are malformed; flags with
               _table_parse_failed=True
    """
    raw = re.sub(r'<[^>]+>', '', raw).strip()
    if not raw:
        return {}

    def clean_keys(obj):
        if isinstance(obj, dict):
            return {k.strip(): clean_keys(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [clean_keys(i) for i in obj]
        return obj

    try:
        return clean_keys(json.loads(raw))
    except Exception:
        pass

    s2 = raw
    s2 = re.sub(r',\s*}', '}', s2)
    s2 = re.sub(r',\s*]', ']', s2)
    s2 = re.sub(r'[\x00-\x1f\x7f]', '', s2)
    s2 = re.sub(r',\s*"[^"]*"\s*(?=[,}])', '', s2)
    try:
        return clean_keys(json.loads(s2))
    except Exception:
        pass

    last_complete = s2.rfind('",')
    if last_complete > 0:
        try:
            return clean_keys(json.loads(s2[:last_complete + 1] + '}'))
        except Exception:
            pass
    try:
        return clean_keys(json.loads(s2))
    except Exception:
        pass

    result = {}
    for m in re.finditer(r'"([^"]+)"\s*:\s*"([^"]*)"', raw):
        key = m.group(1).strip()
        val = m.group(2)
        if key:
            result[key] = val
    if result:
        result['_table_parse_failed'] = True
        return result

    return {}

def evaluate_model(model_path, records, num_beams=3, batch_size=8, label='Model'):
    """
    Load a saved model, run batched inference on records, return per-sample metrics.
    """
    print(f'\nLoading {label} from {model_path}…')
    proc, model = load_eval_model(model_path, device)
    _dtype = next(model.parameters()).dtype   # fp16 after load

    results = []
    t0 = time.time()
    _first_err_printed = False

    for i in tqdm(range(0, len(records), batch_size), desc=f'Eval {label}'):
        chunk   = records[i:i+batch_size]
        images  = [Image.open(os.path.join(IMAGE_BASE_DIR, r['images'][0])).convert('RGB')
                   for r in chunk]
        prompts = [TASK_TOKEN] * len(chunk)

        inp = proc(text=prompts, images=images, return_tensors='pt', padding=True).to(device)
        # Cast pixel_values to model dtype — processor always returns fp32
        inp['pixel_values'] = inp['pixel_values'].to(_dtype)

        try:
            with torch.no_grad():
                out = model.generate(
                    input_ids=inp['input_ids'],
                    pixel_values=inp['pixel_values'],
                    # attention_mask intentionally omitted —
                # Florence-2 rebuilds it after image/text merge
                    max_new_tokens=MAX_LABEL_LEN,
                    num_beams=num_beams,
                )
        except Exception as e:
            if not _first_err_printed:
                print(f'\n⚠ First generate() exception: {type(e).__name__}: {e}')
                _first_err_printed = True
            # Score each item in the batch as zero rather than skipping
            for record in chunk:
                gt = json.loads(record['messages'][1]['content'])
                results.append({
                    'image': record['images'][0],
                    'doc_type': gt.get('Document_Type', 'UNKNOWN'),
                    'gt': gt, 'pred': {},
                    'em': 0.0, 'cer': 1.0, 'fuzzy': 0.0,
                })
            continue

        decoded = proc.batch_decode(out, skip_special_tokens=True)
        for raw, record in zip(decoded, chunk):
            pred = robust_parse(raw)
            gt   = json.loads(record['messages'][1]['content'])
            keys = set(gt) | set(pred)

            em_s  = [1.0 if normalize(gt.get(k)) == normalize(pred.get(k)) else 0.0 for k in keys]
            cer_s = [cer(gt.get(k,''), pred.get(k,'')) for k in keys]
            fz_s  = [fuzz.ratio(normalize(gt.get(k,'')), normalize(pred.get(k,''))) / 100 for k in keys]

            results.append({
                'image':    record['images'][0],
                'doc_type': gt.get('Document_Type', 'UNKNOWN'),
                'gt':  gt, 'pred': pred,
                'em':    float(np.mean(em_s)),
                'cer':   float(np.mean(cer_s)),
                'fuzzy': float(np.mean(fz_s)),
            })

    elapsed = time.time() - t0
    s_per   = elapsed / max(len(records), 1)
    docs_hr = 3600 / s_per

    agg = {
        'label':       label,
        'exact_match': float(np.mean([r['em']    for r in results])),
        'cer':         float(np.mean([r['cer']   for r in results])),
        'fuzzy':       float(np.mean([r['fuzzy'] for r in results])),
        'docs_per_hr': docs_hr,
        'samples':     results,
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return agg

print('evaluate_model() defined ✅')


## 10. Run Evaluations on Both Models

In [ ]:
# Run on the shared validation split (same 406 samples both models never saw)
results_A = evaluate_model(FINAL_FRESH,     val_records, label='A: Fresh 8K')
results_B = evaluate_model(FINAL_WARMSTART, val_records, label='B: WarmStart 8K')

all_results = [results_A, results_B]
print('\nEvaluations complete ✅')


## 11. Head-to-Head Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Text summary ───────────────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'{"Model":<30} {"Exact Match":>12} {"CER":>8} {"Fuzzy":>8} {"Docs/hr":>10}')
print(f'{"-"*70}')
for r in all_results:
    winner_em  = ' ← wins' if r['exact_match'] == max(x['exact_match'] for x in all_results) else ''
    winner_cer = ' ← wins' if r['cer']         == min(x['cer']         for x in all_results) else ''
    print(f'{r["label"]:<30} {r["exact_match"]:>12.3f}{winner_em}')
    print(f'  {"":28} {r["cer"]:>20.3f}{winner_cer}')
    print(f'  {"":28} {r["fuzzy"]:>20.3f}')
    print(f'  {"":28} {r["docs_per_hr"]:>18.0f} docs/hr')
    print()
print('='*70)

# ── Improvement of warm-start over fresh ───────────────────────────────────
em_delta  = results_B['exact_match'] - results_A['exact_match']
cer_delta = results_B['cer']         - results_A['cer']           # negative = better
fz_delta  = results_B['fuzzy']       - results_A['fuzzy']

print(f'Warm-start delta vs Fresh:')
print(f'  EM  : {em_delta:+.3f}  ({"better" if em_delta  > 0 else "worse"})')
print(f'  CER : {cer_delta:+.3f}  ({"better" if cer_delta < 0 else "worse"})')
print(f'  Fuzz: {fz_delta:+.3f}  ({"better" if fz_delta  > 0 else "worse"})')


## 12. Visual Comparison Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Florence-2: Fresh vs Warm-Start Fine-Tuning on 8,125 NSSF Samples',
             fontsize=14, fontweight='bold')

labels  = [r['label'] for r in all_results]
colors  = ['#2196F3', '#FF9800']
metrics = [
    ('Exact Match ↑',  [r['exact_match'] for r in all_results], True),
    ('CER ↓',          [r['cer']         for r in all_results], False),
    ('Fuzzy Match ↑',  [r['fuzzy']       for r in all_results], True),
]

for ax, (title, vals, higher_better) in zip(axes, metrics):
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    best = max(vals) if higher_better else min(vals)
    for bar, val in zip(bars, vals):
        if val == best:
            bar.set_edgecolor('#1a1a1a')
            bar.set_linewidth(2.5)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, min(1.1, max(vals) * 1.25))
    ax.set_xticklabels(labels, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → comparison_chart.png')


## 13. Per-Document-Type Breakdown

In [ ]:
# Which doc types does warm-start improve the most vs fresh?
def per_doctype(result):
    by_type = {}
    for s in result['samples']:
        dt = s['doc_type']
        by_type.setdefault(dt, []).append(s['em'])
    return {dt: np.mean(vals) for dt, vals in by_type.items()}

dt_A = per_doctype(results_A)
dt_B = per_doctype(results_B)

all_types = sorted(set(dt_A) | set(dt_B))

print(f'{"Document Type":<55} {"Fresh EM":>10} {"WarmStart EM":>13} {"Delta":>8}')
print('-' * 90)
for dt in sorted(all_types, key=lambda t: -(dt_B.get(t,0) - dt_A.get(t,0))):
    a = dt_A.get(dt, float('nan'))
    b = dt_B.get(dt, float('nan'))
    d = b - a if not (np.isnan(a) or np.isnan(b)) else float('nan')
    arrow = '▲' if d > 0.02 else ('▼' if d < -0.02 else ' ')
    print(f'{dt:<55} {a:>10.3f} {b:>13.3f} {d:>+7.3f} {arrow}')

print()
print('▲ = warm-start wins by > 0.02 EM  |  ▼ = fresh wins by > 0.02 EM')


## 14. Training Loss Curves

In [ ]:
# ── WandB note ───────────────────────────────────────────────────────────────
# If USE_WANDB=True, interactive loss curves for both runs are already live at:
#   https://wandb.ai/<your-username>/NSSF-Florence2-OCR
# The local matplotlib plots below serve as an offline backup.

def load_loss_history(output_dir):
    state_path = os.path.join(output_dir, 'trainer_state.json')
    if not os.path.exists(state_path):
        for root, dirs, files in os.walk(output_dir):
            if 'trainer_state.json' in files:
                state_path = os.path.join(root, 'trainer_state.json')
                break
    if not os.path.exists(state_path):
        print(f'No trainer_state.json in {output_dir}')
        return [], []
    log   = json.load(open(state_path))['log_history']
    train = [(e['step'], e['loss'])      for e in log if 'loss'      in e]
    val   = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
    return train, val

train_A, val_A = load_loss_history(OUT_FRESH)
train_B, val_B = load_loss_history(OUT_WARMSTART)

if not any([train_A, val_A, train_B, val_B]):
    print('No local trainer_state.json files found yet.')
    print('Train both models first, then re-run this cell.')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Training Loss: Fresh vs Warm-Start (local backup)',
                 fontsize=13, fontweight='bold')

    for ax, title, tr_A, v_A, tr_B, v_B in [
        (ax1, 'Train Loss',      train_A, None,  train_B, None),
        (ax2, 'Validation Loss', None,    val_A, None,    val_B),
    ]:
        if tr_A: ax.plot([s for s,_ in tr_A], [l for _,l in tr_A],
                         color='#2196F3', alpha=0.7, label='A: Fresh')
        if v_A:  ax.plot([s for s,_ in v_A],  [l for _,l in v_A],
                         color='#2196F3', lw=2.5, label='A: Fresh (val)')
        if tr_B: ax.plot([s for s,_ in tr_B], [l for _,l in tr_B],
                         color='#FF9800', alpha=0.7, label='B: Warm-Start')
        if v_B:  ax.plot([s for s,_ in v_B],  [l for _,l in v_B],
                         color='#FF9800', lw=2.5, label='B: Warm-Start (val)')
        ax.set_title(title);  ax.set_xlabel('Step');  ax.set_ylabel('Loss')
        ax.legend();  ax.spines['top'].set_visible(False);  ax.spines['right'].set_visible(False)

    plt.tight_layout()
    out_png = os.path.join(BASE_DIR, 'loss_curves.png')
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Loss curves saved → {out_png}')


## 15. Side-by-Side Prediction Inspector
Change `INSPECT_IDX` to look at any validation sample.


In [ ]:
INSPECT_IDX = 0   # ← change this to inspect different validation samples

item     = val_records[INSPECT_IDX]
gt       = json.loads(item['messages'][1]['content'])
img_path = os.path.join(IMAGE_BASE_DIR, item['images'][0])

print(f'Image   : {item["images"][0]}')
print(f'Doc type: {gt.get("Document_Type", "?")}')

def single_predict(model_path, item):
    proc, model = load_eval_model(model_path, device)
    _dtype = next(model.parameters()).dtype
    img    = Image.open(img_path).convert('RGB')
    inp    = proc(text=TASK_TOKEN, images=img, return_tensors='pt').to(device)
    # Cast pixel_values to model dtype (processor always returns fp32)
    inp['pixel_values'] = inp['pixel_values'].to(_dtype)
    with torch.no_grad():
        out = model.generate(
            input_ids=inp['input_ids'],
            pixel_values=inp['pixel_values'],
            max_new_tokens=MAX_LABEL_LEN,
            num_beams=3,
        )
    raw  = proc.batch_decode(out, skip_special_tokens=True)[0]
    pred = robust_parse(raw)
    del model; gc.collect(); torch.cuda.empty_cache()
    return pred

pred_A = single_predict(FINAL_FRESH,     item)
pred_B = single_predict(FINAL_WARMSTART, item)

# Side-by-side table
all_keys = sorted(set(gt) | set(pred_A) | set(pred_B))
print(f'\n{"Field":<35} {"Ground Truth":<30} {"Model A Fresh":<30} {"Model B WarmStart":<30}')
print('-' * 128)
for k in all_keys:
    g = str(gt.get(k, ''))[:28]
    a = str(pred_A.get(k, ''))[:28]
    b = str(pred_B.get(k, ''))[:28]
    a_ok = '✓' if normalize(gt.get(k,'')) == normalize(pred_A.get(k,'')) else '✗'
    b_ok = '✓' if normalize(gt.get(k,'')) == normalize(pred_B.get(k,'')) else '✗'
    print(f'{k:<35} {g:<30} {a_ok} {a:<28} {b_ok} {b:<28}')

from IPython.display import display
display(Image.open(img_path).convert('RGB'))


---
## 16. Interpreting Results & Choosing a Winner

### What to expect
The **warm-start model should converge faster** (lower loss at epoch 1–2) because the 87-sample model already learned:
- NSSF document layout and field positions
- Kenyan-specific fonts, abbreviations, and stamps
- Rough JSON output structure

By epoch 4–5 both models often reach similar final accuracy on well-represented doc types.  
The warm-start tends to stay ahead on **rare or difficult types** (e.g. Loss Certificates, handwritten EFT forms) where 8K samples alone may not be enough.

### Decision table
| Situation | Use |
|-----------|-----|
| Warm-start EM > Fresh EM by ≥ 0.03 | Deploy Model B (warm-start) |
| Similar EM but warm-start CER lower | Deploy Model B |
| Fresh wins on most common doc types | Deploy Model A (simpler lineage) |
| Both underperform (< 0.60 EM) | Add more epochs or training data |

### If warm-start wins on specific doc types only
Create **two specialised adapters** — one warm-started for rare docs, one fresh for common docs — and route by `Document_Type` during inference.

### Production deployment
```python
from transformers import AutoProcessor, AutoModelForCausalLM
import torch, json, re

# Swap WINNER_PATH for whichever model scored better
WINNER_PATH = 'florence2_warmstart_8k_final'   # or florence2_fresh_8k_final

proc  = AutoProcessor.from_pretrained(WINNER_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    WINNER_PATH, trust_remote_code=True, torch_dtype=torch.float16,
).to('cuda').eval()

def robust_parse(raw):
    raw = re.sub(r'<[^>]+>', '', raw).strip()
    if not raw: return {}
    def clean_keys(obj):
        if isinstance(obj, dict): return {k.strip(): clean_keys(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean_keys(i) for i in obj]
        return obj
    try: return clean_keys(json.loads(raw))
    except: pass
    s2 = re.sub(r',\s*}', '}', re.sub(r',\s*]', ']', raw))
    s2 = re.sub(r',\s*"[^"]*"\s*(?=[,}])', '', s2)
    try: return clean_keys(json.loads(s2))
    except: pass
    last = s2.rfind('",')
    if last > 0:
        try: return clean_keys(json.loads(s2[:last+1] + '}'))
        except: pass
    result = {m.group(1).strip(): m.group(2) for m in re.finditer(r'"([^"]+)"\s*:\s*"([^"]*)"', raw)}
    return dict(result, _table_parse_failed=True) if result else {}

def extract(image_path):
    img = Image.open(image_path).convert('RGB')
    inp = proc(text='<Extract_NSSF_Data>', images=img, return_tensors='pt').to('cuda')
    _dtype = next(model.parameters()).dtype
    inp['pixel_values'] = inp['pixel_values'].to(_dtype)
    with torch.no_grad():
        out = model.generate(
            input_ids=inp['input_ids'],
            pixel_values=inp['pixel_values'],
            # attention_mask intentionally omitted — Florence-2 rebuilds it internally
            max_new_tokens=640,
            num_beams=3,
        )
    raw = proc.batch_decode(out, skip_special_tokens=True)[0]
    return robust_parse(raw)
```
